In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

### Pre-processing

In [8]:
# 1. converting to lowercase
df["review"] = df["review"].str.lower()

In [9]:
# remove the URLs
import re
def remove_url(text) :
    text = re.sub(r"http\S+","", text)
    return text

df["review"] = df["review"].apply(remove_url)

In [ ]:
# Remove punctuation
def remove_punc(text) :
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # ^ -> keep [] all value
    return text

df["review"] = df["review"].apply(remove_punc)

In [11]:
# Remove HTML
def remove_html(text) :
    text = re.sub(r"<.*?>", "", text) # ? -> non greedy manner. stop when find the >
    return text

df["review"] = df["review"].apply(remove_html)

In [ ]:
# Remove the stopswords
import nltk
nltk.download("punkt")
nltk.download("punkt_tb")
nltk.download("stopwords")

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
def remove_stopwords(text) :
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens :
        if word in stop_words :
            text = text.replace(word,"")
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [ ]:
df.head()

In [ ]:
# Stemming
# running -> run
# played -> play
# ProterStemming

from nltk.stem import ProterStemming


In [ ]:
def stemming(text) :
    ps = ProterStemming()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens :
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [ ]:
# Encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [ ]:
y = df["sentiment"]

In [ ]:
# Vectorization
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)
X = tf.fit_transform(df["review"])

## Dataset & DataLoader

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
X_train = X_train.toarray() # X_train was a sparse matrix so we have to convert it into numpy array before TensorDataset
X_test = X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, shuffle=True, batch_size = 64)
test_loader = DataLoader(test_set, shuffle=True, batch_size = 64)

### Build RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module) :
    def __init__(self, input_size, hidden_size=128, num_layers = 1) :
        super().__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x) :
        # optional -> shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _, = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps -> (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out
        

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training the RNN

In [ ]:
epochs = 10
for epoch in range(epochs) :
    model.train()

    for Xb, yb in train_loader :
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add single direction

        outputs = model(Xb) # (batch_size, 1)
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) -> probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch}/{epochs} and loss = {loss.item()}")

In [ ]:
# evaluation 
model.eval()

with torch.no_grad() :
    correct_vals = 0
    total_vals = 0

    for Xb, yb in test_loader :
        Xb = Xb.unsqueeze()

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals +=size(0)
        correct_vals += (predicted == yb).sum().item()
    print(f"accuracy = {correct_vals/total_vals}")